In [1]:
import os
import random

In [2]:
# Define the folder containing the audio files
audio_folder = r"AudioFiles/085"

# Get a list of all WAVE files in the folder
all_songs = [f for f in os.listdir(audio_folder) if f.endswith(".wav")]

In [3]:
k = 5  # songs to rank
random.shuffle(all_songs)

max_tests = len(all_songs) // (k + 1)
num_tests = min(10, max_tests)

test_sets = []
idx = 0

for _ in range(num_tests):
    reference = all_songs[idx]
    candidates = all_songs[idx + 1: idx + 1 + k]
    test_sets.append((reference, candidates))
    idx += k + 1

In [4]:
# Background image path
background_image = "background.png"

In [5]:
SUPABASE_URL = "https://zpvhnhnmjitkskhrhalp.supabase.co"
SUPABASE_ANON_KEY = "sb_publishable_jrzABfRbbZN4BNrUmABKag_b3LlgY18"

In [10]:
html_content = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Music Similarity Test</title>

    <style>
        body {{
            font-family: Helvetica, Arial, sans-serif;
            background: url('{background_image}') no-repeat center center fixed;
            background-size: cover;
            color: white;
            max-width: 900px;
            margin: 0 auto;
            padding: 20px;
        }}

        input[type="text"] {{
            padding: 6px;
            width: 220px;
            margin-top: 4px;
            margin-bottom: 16px;
        }}

        audio {{
            display: block;
            margin-bottom: 12px;
            width: 100%;
        }}

        .test-block {{
            margin-bottom: 30px;
            padding: 16px;
            background: rgba(0, 0, 0, 0.35);
            border-radius: 10px;
        }}

        .sortable-list {{
            list-style: none;
            padding: 0;
            margin: 0;
        }}

        .sortable-item {{
            margin-bottom: 12px;
            padding: 12px;
            background: rgba(255, 255, 255, 0.08);
            border-radius: 8px;
            cursor: grab;
        }}

        .sortable-item:active {{
            cursor: grabbing;
        }}

        .rank-label {{
            font-weight: bold;
            margin-bottom: 8px;
        }}

        .submit-row {{
            margin-top: 24px;
        }}
    </style>

    <script src="https://cdn.jsdelivr.net/npm/@supabase/supabase-js@2"></script>
    <script src="https://cdn.jsdelivr.net/npm/sortablejs@latest/Sortable.min.js"></script>

    <script>
        const SUPABASE_URL = "{SUPABASE_URL}";
        const SUPABASE_ANON_KEY = "{SUPABASE_ANON_KEY}";
        const supabaseClient = supabase.createClient(SUPABASE_URL, SUPABASE_ANON_KEY);

        document.addEventListener("DOMContentLoaded", function () {{
            for (let i = 0; i < {num_tests}; i++) {{
                new Sortable(document.getElementById("list_" + i), {{
                    animation: 150
                }});
            }}
        }});

        async function saveResults(event) {{
            event.preventDefault();

            const consentGiven = document.getElementById("consent").checked;
            if (!consentGiven) {{
                alert("You must agree to participate before submitting.");
                return;
            }}

            const participantCode = document.getElementById("participant_code")?.value.trim() || null;

            let results = [];

            for (let i = 0; i < {num_tests}; i++) {{
                const reference = document.getElementById("reference_" + i).dataset.song;
                const list = document.getElementById("list_" + i);
                const items = list.children;

                let ranking = [];

                for (let rank = 0; rank < items.length; rank++) {{
                    const song = items[rank].dataset.song;
                    ranking.push({{
                        song: song,
                        rank: rank + 1
                    }});
                }}

                results.push({{
                    test: i + 1,
                    reference_song: reference,
                    ranking: ranking
                }});
            }}

            const submitBtn = document.getElementById("submitBtn");
            submitBtn.disabled = true;
            submitBtn.value = "Submitting...";

            const {{ error }} = await supabaseClient
                .from("listening_results")
                .insert([{{
                    participant_code: participantCode,
                    consent: true,
                    test_version: "v3_dragdrop_ranking",
                    responses: results
                }}]);

            submitBtn.disabled = false;
            submitBtn.value = "Save Results";

            if (error) {{
                console.error("Supabase insert error:", error);
                alert("Submission failed. Check console.");
                return;
            }}

            alert("Thank you. Your responses were saved.");
            document.querySelector("form").reset();
        }}
    </script>
</head>
<body>
    <h1>Music Similarity Test</h1>

    <p>
        Please drag the songs into order from <b>most similar (top)</b> to <b>least similar (bottom)</b>
        relative to the reference song.
    </p>

    <p>
        Participant code:<br>
        <input type="text" id="participant_code" placeholder="e.g. P001">
    </p>

    <form onsubmit="saveResults(event);">
"""

In [11]:
for i, (reference_song, candidates) in enumerate(test_sets):
    shuffled_candidates = candidates.copy()
    random.shuffle(shuffled_candidates)

    html_content += f"""
        <div class="test-block">
            <h3>Test {i + 1}</h3>

            <p><b>Reference Song:</b></p>
            <audio controls id="reference_{i}" data-song="{reference_song}">
                <source src="AudioFiles/085/{reference_song}" type="audio/wav">
            </audio>

            <p>
                Drag the songs below so that the <b>top</b> one is the most similar
                and the <b>bottom</b> one is the least similar.
            </p>

            <ul id="list_{i}" class="sortable-list">
    """

    for j, song in enumerate(shuffled_candidates):
        html_content += f"""
                <li class="sortable-item" id="song_{i}_{j}" data-song="{song}">
                    <div class="rank-label">Song {j + 1}</div>
                    <audio controls>
                        <source src="AudioFiles/085/{song}" type="audio/wav">
                    </audio>
                </li>
        """

    html_content += """
            </ul>
        </div>
    """

In [12]:
html_content += """
        <p>
            <input type="checkbox" id="consent" required>
            I agree to participate in this study and allow my responses to be used for research purposes.
        </p>

        <div class="submit-row">
            <input type="submit" id="submitBtn" value="Save Results">
        </div>
    </form>

    <p style="font-size: 12px; margin-top: 20px;">
        This experiment is based on research from
        <a href="https://arxiv.org/abs/1612.01840" target="_blank" style="color: white;">
            arXiv:1612.01840
        </a>.
    </p>

    <p style="font-size: 12px;">
        The metadata is released under the Creative Commons Attribution 4.0 International License (CC BY 4.0).
        We do not hold the copyright on the audio and distribute it under the license chosen by the artist.
        The dataset is meant for research purposes.
    </p>
</body>
</html>
"""

In [13]:
with open("index.html", "w", encoding="utf-8") as f:
    f.write(html_content)

print("index.html generated successfully.")

index.html generated successfully.
